# 12.2 规划 (Planning)

规划使Agent能将复杂任务分解为可执行的步骤序列。

本节涵盖：任务分解、规划算法、反思与修正

## 1. 任务分解与规划算法

**任务分解**：将复杂任务拆分为子任务序列。
- **CoT（Chain of Thought）**：逐步推理，每步输出中间结果
- **ReWOO**：先规划所有步骤，再依次执行

**规划算法**：
- **Tree of Thought**：探索多条推理路径，选择最优
- **MCTS**：蒙特卡洛树搜索，平衡探索与利用

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class TreeOfThought(nn.Module):
    def __init__(self, d=64, n_branches=3, max_depth=3):
        super().__init__()
        self.n_branches = n_branches
        self.max_depth = max_depth
        self.value_net = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
        self.state_net = nn.Linear(d, d)

    def expand(self, state):
        children = []
        for _ in range(self.n_branches):
            noise = torch.randn_like(state) * 0.3
            child = self.state_net(state + noise)
            value = self.value_net(child)
            children.append((child, value.item()))
        return children

    def search(self, initial_state):
        best_path = []
        best_value = 0
        current = initial_state
        for depth in range(self.max_depth):
            children = self.expand(current)
            best_child, best_val = max(children, key=lambda x: x[1])
            best_path.append((depth, best_val))
            current = best_child
            best_value = best_val
        return best_path, best_value

tot = TreeOfThought()
initial = torch.randn(1, 64)
path, final_value = tot.search(initial)

print('=== Tree of Thought ===')
print(f'Branches: {tot.n_branches}, Max depth: {tot.max_depth}')
print(f'\nBest path found:')
for depth, value in path:
    print(f'  Depth {depth}: value={value:.4f}')
print(f'Final value: {final_value:.4f}')
print(f'\nKey: ToT explores multiple reasoning paths, selecting the most promising.')

## 2. 反思与修正 (Reflection)

**Reflexion**：Agent执行任务后进行自我反思，将反思结果作为下次尝试的额外输入，迭代改进。

**Self-Refine**：模型对自身输出进行多轮自我批评和修正，逐步提升输出质量。

**核心思想**：失败不是终点，而是学习的机会。通过反思将失败经验转化为改进策略。

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class ReflexionAgent(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.action_net = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, 4))
        self.reflect_net = nn.Sequential(nn.Linear(d * 2, 32), nn.ReLU(), nn.Linear(32, d))
        self.evaluate_net = nn.Sequential(nn.Linear(d, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())

    def act(self, state):
        return self.action_net(state)

    def reflect(self, state, feedback):
        combined = torch.cat([state, feedback], dim=-1)
        return self.reflect_net(combined)

    def evaluate(self, state):
        return self.evaluate_net(state)

agent = ReflexionAgent()
state = torch.randn(1, 64)

print('=== Reflexion ===')
for trial in range(3):
    action = agent.act(state)
    score = agent.evaluate(state)
    print(f'Trial {trial+1}: action={action.argmax().item()}, score={score.item():.4f}')
    if score > 0.8:
        print(f'  Success!')
        break
    feedback = torch.randn(1, 64)
    reflection = agent.reflect(state, feedback)
    state = state + 0.1 * reflection
    print(f'  Reflecting... adjusting state')

print(f'\nKey: Reflexion uses failure feedback to improve future attempts.')
print(f'Each trial incorporates reflections from previous failures.')